# AutoDock Vina Protein-Ligand Docking

![AutoDock Vina Protein-Ligand Docking](https://proto-bio.github.io/proto-assets/images/tool/vina/hero.png)

This notebook includes an example of using `run_vina_docking` to redock imatinib into the c-Abl kinase structure from PDB 1IEP, and a second example of using it to screen four clinical BCR-ABL inhibitors against that same pocket in a single call.


In [ ]:
from proto_tools.utils.notebook_docs import display_available_tools, display_api_reference, display_doc_link, display_docs_section, display_overview

display_doc_link("vina")
display_overview("vina")
display_docs_section("vina", "Background")
display_available_tools("vina")

## Redock imatinib into c-Abl

The receptor fixture is the protein from PDB 1IEP. The reference ligand fixture is retained only to demonstrate automatic box construction; the docking ligand itself is supplied as a protonated SMILES string.

In [ ]:
from pathlib import Path

from proto_tools import (
    Structure,
    VinaDockingConfig,
    VinaDockingInput,
    VinaReferenceLigandBox,
    run_vina_docking,
)
from proto_tools.tools.molecular_docking.vina import visualize_search_box
from proto_tools.tools.molecular_docking.vina.vina_docking import (
    example_input,
    example_reference_ligand,
)

display_api_reference("vina", "input", "run_vina_docking")
display_api_reference("vina", "config", "run_vina_docking")
display_api_reference("vina", "output", "run_vina_docking")

In [ ]:
inputs = example_input()

print("Receptor chains:", inputs.receptor.get_chain_ids())
print("Ligand SMILES:", [ligand.smiles for ligand in inputs.ligands])
print("Explicit search box:", inputs.search_box)

reference_box = VinaReferenceLigandBox(reference_ligand=example_reference_ligand(), padding=4.0)
print("Reference-derived box:", reference_box.resolve())


### Check the search box before running

A box that is misplaced or too small does not raise — Vina searches wherever it is pointed and returns confident-looking scores either way, so the failure is invisible in the numbers. `visualize_search_box` draws the receptor with the resolved box, and passing the config also reports the affinity-map grid size, so an over-fine grid surfaces here rather than as an error mid-run.

Both ways of specifying the box are shown below. The explicit `VinaSearchBox` is the 20 A cube typed into `example_input()`. The `VinaReferenceLigandBox` derives its cage from imatinib's crystallographic coordinates instead, so it comes out fitted to the ligand rather than cubic, and that reference ligand is drawn inside the cage — it is the one ligand here carrying real coordinates in the receptor frame, which is what makes it useful for confirming the box sits on the pocket.


In [ ]:
config = VinaDockingConfig(
    exhaustiveness=8,
    num_poses=5,
    energy_range=4.0,
    cpu=1,
    seed=7,
)

# Explicit box: a 20 A cube, nothing to draw inside it.
visualize_search_box(inputs, config)

# Reference-ligand box: fitted to imatinib, which is drawn in the cage.
reference_inputs = VinaDockingInput(
    receptor=inputs.receptor,
    ligands=inputs.ligands,
    search_box=reference_box,
)
visualize_search_box(reference_inputs, config)


In [ ]:
result = run_vina_docking(inputs, config)
result


In [ ]:
import pandas as pd

# `results` holds one entry per input ligand, in input order.
docked = result.results[0]
pd.DataFrame(
    [
        {
            "rank": pose.rank,
            "affinity_kcal_mol": pose.metrics.affinity,
            "rmsd_from_best_lower_bound_angstrom": pose.metrics.rmsd_lower_bound,
            "rmsd_from_best_upper_bound_angstrom": pose.metrics.rmsd_upper_bound,
        }
        for pose in docked.poses
    ]
)


## Inspect and export the top-ranked pose

In [ ]:
import py3Dmol

view = py3Dmol.view(width=900, height=600)
view.addModel(inputs.receptor.structure_pdb, "pdb")
view.setStyle({"model": 0}, {"cartoon": {"color": "spectrum"}})
view.addModel(docked.poses[0].sdf, "sdf")
view.setStyle({"model": 1}, {"stick": {"colorscheme": "greenCarbon"}})
view.zoomTo({"model": 1})
view.show()


In [ ]:
output_dir = Path("example_output")
result.export("imatinib_poses", output_dir, file_format="sdf")
result.export("imatinib_scores", output_dir, file_format="csv")
print(output_dir.resolve())

## Screen several inhibitors in one call

`ligands` is the per-item field, so a whole set goes through one call and comes back as one result per ligand, in input order. Receptor preparation is the expensive setup step and depends only on the receptor, so it runs once and is reused across the set.

1IEP is the imatinib-bound structure, so its activation loop sits in the DFG-out (inactive) conformation. Imatinib, nilotinib, and ponatinib are all type-II binders that occupy that conformation, so this receptor suits them. Dasatinib is a type-I binder that engages the active conformation instead, so a weaker score here reflects the conformational mismatch rather than lower potency. A docking score is always conditioned on the receptor conformation you supply.


In [ ]:
inhibitors = {
    "imatinib": "Cc1ccc(NC(=O)c2ccc(CN3CC[NH+](C)CC3)cc2)cc1Nc1nccc(-c2cccnc2)n1",
    "nilotinib": "Cc1ccc(cc1Nc1nccc(n1)-c1cccnc1)C(=O)Nc1cc(cc(c1)C(F)(F)F)n1cc(C)nc1",
    "ponatinib": "Cc1ccc(cc1C#Cc1cnc2n1nccc2)C(=O)Nc1ccc(CN2CCN(C)CC2)c(c1)C(F)(F)F",
    "dasatinib": "Cc1nc(Nc2ncc(s2)C(=O)Nc2c(C)cccc2Cl)cc(n1)N1CCN(CCO)CC1",
}

screen = run_vina_docking(
    VinaDockingInput(
        receptor=inputs.receptor,
        ligands=list(inhibitors.values()),
        search_box=inputs.search_box,
    ),
    VinaDockingConfig(exhaustiveness=8, num_poses=3, energy_range=4.0, cpu=1, seed=7),
)

pd.DataFrame(
    [
        {
            "compound": name,
            "binding_mode": mode,
            "best_affinity_kcal_mol": docked.poses[0].metrics.affinity,
            "poses_returned": len(docked.poses),
        }
        for (name, mode), docked in zip(
            [("imatinib", "type-II"), ("nilotinib", "type-II"),
             ("ponatinib", "type-II"), ("dasatinib", "type-I")],
            screen.results,
            strict=True,
        )
    ]
).sort_values("best_affinity_kcal_mol").reset_index(drop=True)


## Fetch the receptor from the PDB

`Structure.from_rcsb` retrieves a deposited entry directly, which is the usual starting point outside this notebook. Selecting the receptor chain still leaves the crystallographic waters, chloride ions, and the co-crystallised imatinib in place, so those non-polymer residues must be stripped before docking: Meeko rejects residues it cannot parameterize unless `allow_bad_residues=True`, which deletes them, lists them in `ignored_receptor_residues`, and warns. The bundled fixture used above is exactly this chain with those residues removed — and redocking imatinib into a pocket that still contains imatinib would not be a meaningful validation.

In [ ]:
deposited = Structure.from_rcsb("1IEP")
receptor_chain_a = deposited.select_chain("A")

non_polymer = sorted(
    {line[17:20].strip() for line in receptor_chain_a.structure.splitlines() if line.startswith("HETATM")}
)
print("Chains in the deposited entry:", deposited.get_chain_ids())
print("Non-polymer residues still present in chain A:", non_polymer)